# 🔭 DepthScope: Monocular Depth Estimation
### Predicting 3D Scene Geometry from a Single 2D RGB Image

Welcome to **DepthScope**! In this hands-on computer vision experiment, we explore **Monocular Depth Estimation (MDE)** using a state-of-the-art pretrained deep learning model.

---

### 🧠 What is Monocular Depth Estimation?
Humans perceive depth and 3D structure through a combination of binocular vision (two eyes providing stereo disparity) and monocular visual cues. Even when looking at a single 2D photograph with one eye closed, you can intuitively discern what is close and what is far away by recognizing:
- **Perspective and vanishing lines**: Converging parallel lines signal receding distance.
- **Occlusion**: Foreground objects visually overlap and block background objects.
- **Texture gradients**: Surface textures appear denser and finer as distance increases.
- **Relative size**: Known objects (people, cars, trees) subtend smaller visual angles when situated farther away.

**Monocular Depth Estimation (MDE)** is the computer vision task of predicting a dense depth map from a single 2D RGB photograph. Rather than requiring multi-view stereo rigs, LiDAR sensors, or Time-of-Flight (ToF) cameras, MDE relies entirely on learning these visual cues from vast image datasets.

---

### 🤖 The Pretrained Model: Depth Anything V2 (Small)
In this experiment, we utilize **Depth Anything V2 Small** (`depth-anything/Depth-Anything-V2-Small-hf`), developed by ByteDance and HKU:
- **Vision Transformer (ViT) Backbone**: Extracts multi-scale visual tokens and reasons over long-range spatial context.
- **Zero-Shot Robustness**: Trained on massive synthetic and unlabeled real-world datasets, enabling it to generalize to diverse indoor and outdoor scenes.
- **Lightweight & Fast**: The `Small` checkpoint delivers high-detail relative depth predictions with fast inference on Google Colab GPUs (and smooth fallback on CPU).
- **Relative Depth Output**: Predicts per-pixel relative disparity (closer vs. farther) across the entire scene.

Let's step through the pipeline! 🚀

In [ ]:
# install transformers and accelerate quietly so colab stays clean
!pip install -q transformers accelerate

In [ ]:
# pull in all the standard tools we need
import os
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline

# check colab environment for upload/download utils
try:
    from google.colab import files
    COLAB_ENV = True
except ImportError:
    COLAB_ENV = False
    print("ℹ️ Running outside Colab environment - local file paths will be used.")

# grab cuda if we got it, else chill on cpu
device = 0 if torch.cuda.is_available() else -1
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"⚡ Compute device: {device_name} (device id: {device})")

In [ ]:
# load up depth anything v2 small once so we don't reload every time
model_id = "depth-anything/Depth-Anything-V2-Small-hf"

print(f"📦 Loading pretrained depth estimation model: {model_id}...")

# instantiate hugging face depth-estimation pipeline
pipe = pipeline(
    task="depth-estimation",
    model=model_id,
    device=device
)

print("✅ Pipeline loaded and ready to cook!")

In [ ]:
# prompt user to upload a single photo
uploaded_image_path = None

if COLAB_ENV:
    print("📁 Please upload a photograph (supported formats: JPG, JPEG, PNG)...")
    uploaded = files.upload()
    
    # make sure user actually dropped an image
    if not uploaded:
        raise ValueError("❌ No file uploaded! Please re-run this cell and choose a valid image file.")
    
    # grab the first uploaded file name
    uploaded_file_name = list(uploaded.keys())[0]
    
    # validate file extension
    valid_exts = (".jpg", ".jpeg", ".png")
    if not uploaded_file_name.lower().endswith(valid_exts):
        raise ValueError(f"❌ Unsupported file format for '{uploaded_file_name}'. Supported formats: JPG, JPEG, PNG.")
        
    uploaded_image_path = uploaded_file_name
    print(f"🎉 Upload successful: {uploaded_image_path}")
else:
    # local testing fallback
    sample_path = "sample.jpg"
    if os.path.exists(sample_path):
        uploaded_image_path = sample_path
        print(f"📁 Using local fallback image: {uploaded_image_path}")
    else:
        raise FileNotFoundError("❌ No image found. In Colab, run files.upload(). Locally, place 'sample.jpg' in the directory.")

In [ ]:
# open image with pil and force rgb so no weird channels
try:
    raw_image = Image.open(uploaded_image_path)
    img_rgb = raw_image.convert("RGB")
    
    # grab and display image dimensions
    img_width, img_height = img_rgb.size
    print(f"🖼️ Image successfully opened!")
    print(f"📐 Dimensions: {img_width} x {img_height} pixels (Width x Height)")
    print(f"🎨 Image Color Mode: {img_rgb.mode}")
except Exception as e:
    raise RuntimeError(f"❌ Could not open or process image '{uploaded_image_path}': {e}")

In [ ]:
# clock how fast the model cooks
start_time = time.perf_counter()

try:
    # run inference through the depth pipeline without extra fine-tuning
    with torch.no_grad():
        output = pipe(img_rgb)
    end_time = time.perf_counter()
    
    inference_time_ms = (end_time - start_time) * 1000.0
    print(f"⚡ Inference completed in {inference_time_ms:.2f} ms ({end_time - start_time:.4f} seconds)")
    print(f"📊 Original input size: {img_width} x {img_height}")
except Exception as e:
    raise RuntimeError(f"❌ Depth inference failed: {e}")

In [ ]:
# safely extract depth and normalize to [0, 1]
# check if raw predicted_depth tensor is provided
if "predicted_depth" in output and output["predicted_depth"] is not None:
    raw_tensor = output["predicted_depth"]
    
    # move tensor to cpu and strip extra batch dims
    if hasattr(raw_tensor, "detach"):
        raw_depth_np = raw_tensor.detach().cpu().numpy().squeeze()
    else:
        raw_depth_np = np.array(raw_tensor).squeeze()
        
    # safe min-max normalization to dodge division by zero
    d_min = float(raw_depth_np.min())
    d_max = float(raw_depth_np.max())
    
    if d_max - d_min > 1e-6:
        depth_normalized = (raw_depth_np - d_min) / (d_max - d_min)
    else:
        depth_normalized = np.zeros_like(raw_depth_np, dtype=np.float32)
        
    # resize normalized depth map back to match original image size for crisp display
    depth_map_pil = Image.fromarray((depth_normalized * 255.0).astype(np.uint8)).resize(
        img_rgb.size, resample=Image.Resampling.BILINEAR
    )
    depth_display = np.array(depth_map_pil) / 255.0
else:
    # fallback to the pre-rendered pil depth image
    depth_map_pil = output["depth"].convert("L")
    depth_display = np.array(depth_map_pil, dtype=np.float32) / 255.0

# plot side-by-side: original, grayscale depth, colorized depth map
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

# 1. original photograph
axes[0].imshow(img_rgb)
axes[0].set_title(f"1. Original Photograph\n({img_width} x {img_height})", fontsize=13, fontweight="bold")
axes[0].axis("off")

# 2. grayscale relative depth map
axes[1].imshow(depth_display, cmap="gray")
axes[1].set_title("2. Grayscale Depth Map\n(Brighter = Closer, Darker = Farther)", fontsize=13, fontweight="bold")
axes[1].axis("off")

# 3. colorized relative depth map with colorbar
im_color = axes[2].imshow(depth_display, cmap="inferno")
axes[2].set_title("3. Colorized Relative Depth Map\n(Inferno Colormap)", fontsize=13, fontweight="bold")
axes[2].axis("off")

# add colorbar to show relative scale
cbar = fig.colorbar(im_color, ax=axes[2], fraction=0.046, pad=0.04)
cbar.set_label("Relative Disparity / Depth (Normalized [0, 1])", fontsize=11)

plt.show()

print("💡 Note: Relative depth values represent normalized disparity (closer vs farther), not physical metres or centimetres.")